# Lab 03 — Getting started with Hugging Face

Theo đề bài: suy luận cảm xúc bằng mô hình pretrained và fine-tune mô hình phân loại nhị phân bằng `Trainer`.

Notebook dùng chung mã nguồn với CLI. Chưa có kết quả full training được chèn sẵn: mọi metric, bảng và biểu đồ bên dưới được tạo khi bạn thực thi. Không dùng lại số liệu IMDB từ bài cũ.

**Chuẩn bị:** Python 3.12, cài `Lab3/requirements.txt`, chọn kernel tương ứng; bật Internet để tải Hugging Face. Bài 2 khuyến nghị Kaggle/Colab GPU. Xem README để cài PyTorch đúng CUDA. Không chạy pip tự động trong notebook để tránh thay đổi kernel ngoài ý muốn.

In [ ]:
from pathlib import Path
import sys
# Works from repository root, Lab3/, or Lab3/notebooks/.
LAB = next((p / "Lab3" if (p / "Lab3/src").is_dir() else p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "Lab3/src").is_dir() or (p / "src/workflow.py").is_file()), None)
if LAB is None:
    raise RuntimeError("Open this notebook inside a cloned Lab-Deep-Learning repository")
sys.path.insert(0, str(LAB))
import json
import torch
from IPython.display import display, Image, Markdown
from src.workflow import read_config, seed_all, exercise1, prepare_data, finetune
config = read_config()
seed_all(config["seed"])
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print(json.dumps(config, indent=2))

## Exercise 1 — Pretrained sentiment analysis

Sử dụng DistilBERT đã fine-tune trên SST-2. Tokenizer WordPiece biến văn bản thành ID; `[CLS]` mở đầu, `[SEP]` kết thúc; `attention_mask` đánh dấu token thật và padding. `pipeline` đóng gói tokenizer → model → softmax → nhãn.

Bước sau tải mô hình thật, chạy cả hai cách trên cùng câu và kiểm tra nhãn/xác suất khớp trong sai số `1e-5`.

In [ ]:
exercise1_run = exercise1(config)
print("Run artifacts:", exercise1_run)

In [ ]:
breakdown = json.loads((exercise1_run / "tokenizer_breakdown.json").read_text())
import pandas as pd
display(pd.DataFrame({k: breakdown[k] for k in ["tokens", "input_ids", "attention_mask"]}))
predictions = json.loads((exercise1_run / "predictions.json").read_text())
display(pd.DataFrame(predictions["manual"]))

### Tự kiểm tra
- ID và mask có cùng độ dài không? Tại sao `[CLS]` và `[SEP]` có mask bằng 1?
- Nhãn từ pipeline có khớp cách suy luận thủ công không?
- Thêm câu có phủ định hoặc cấu trúc nhượng bộ; quan sát độ tin cậy. Confidence không đảm bảo dự đoán đúng.